# Data Copilot — eval walkthrough

This notebook is the **3-cell version** of `./scripts/dev.sh eval`. It is the fastest way to see *what the eval harness does* without learning the CLI.

What you'll do:

1. **Setup**: load the case set and confirm the agent boots locally.
2. **Run an A/B**: schema RAG on vs off, full case set, ~30 LLM calls, ~1–2 min.
3. **Visualise**: per-category success rate as a grouped bar chart so the JOIN improvement and the simple-case parity are visible at a glance.

## Prerequisites

* `docker compose up postgres` — Postgres + pgvector running on `localhost:5432`.
* `.env` with `DEEPSEEK_API_KEY` and `SILICONFLOW_API_KEY` filled in (run `./scripts/make-env.sh` if you haven't).
* `uv sync --all-extras` — Jupyter is in the `dev` extras already.

If the eval CLI works, this notebook will work.

> See [ADR 0007](../docs/decisions/0007-eval-methodology.md) for the "why deterministic graders, why not RAGAS, why comparative" rationale.

## Cell 1 — Load cases and verify the agent imports cleanly

We don't run any LLM in this cell; just check that the package and the case file are reachable.

In [ ]:
import sys
from pathlib import Path

# When running from the repo's notebooks/ folder, copilot lives one
# level up under apps/api. Prepend it to sys.path so imports resolve.
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "apps" / "api"))

from copilot.eval.cases import DEFAULT_CASES_PATH, load_cases
from copilot.eval.config import BASELINE_FULL, WITHOUT_SCHEMA_RAG

cases = load_cases(DEFAULT_CASES_PATH)

from collections import Counter
by_category = Counter(c.category for c in cases)
print(f"Loaded {len(cases)} cases from {DEFAULT_CASES_PATH.name}:")
for cat, n in sorted(by_category.items()):
    print(f"  {cat:<22} {n}")

## Cell 2 — Run the schema_rag A/B

Two passes over the full case set: once with `retrieve_schema_node` short-circuited to dump the full DDL (week-2 baseline), once with the week-3 retriever active.

**Cost**: ~30 LLM calls × ¥0.01 ≈ ¥0.3, takes 1–2 min depending on DeepSeek latency.

**The hypothesis** (from ADR 0007): RAG should help **JOIN** cases substantially while leaving simpler categories unchanged.

In [ ]:
import asyncio
from copilot.eval.experiments import run_schema_rag_ab

comparison = await run_schema_rag_ab(cases)  # noqa: F821 — Jupyter top-level await

print(f"baseline ({comparison.baseline.config.label}): "
      f"{comparison.baseline.passed}/{comparison.baseline.total} passed "
      f"({comparison.baseline.success_rate * 100:.1f}%)")
print(f"treatment ({comparison.treatment.config.label}): "
      f"{comparison.treatment.passed}/{comparison.treatment.total} passed "
      f"({comparison.treatment.success_rate * 100:.1f}%)")
print(f"\nDelta: success_rate {comparison.success_rate_delta * 100:+.1f} pp, "
      f"avg_attempts {comparison.avg_attempts_delta:+.2f}, "
      f"avg_total_tokens {comparison.avg_total_tokens_delta:+.0f}")

## Cell 3 — Visualise the per-category breakdown

Grouped bar by category. The interesting bars are wherever the orange (RAG on) is meaningfully taller than the blue (RAG off). Empirically, that's `join` and (sometimes) `follow_up`.

Run `./scripts/dev.sh eval` for the full markdown report (auto-saved to `docs/eval/`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for label, result in [
    ("RAG off (baseline)", comparison.baseline),  # noqa: F821
    ("RAG on (treatment)", comparison.treatment),  # noqa: F821
]:
    for category, stats in result.by_category().items():
        rows.append(
            {
                "variant": label,
                "category": category,
                "success_rate": stats["success_rate"] * 100,
                "n": int(stats["n"]),
            }
        )
df = pd.DataFrame(rows)
pivot = df.pivot(index="category", columns="variant", values="success_rate").sort_index()

ax = pivot.plot.bar(figsize=(10, 4), edgecolor="#333", width=0.75)
ax.set_title("schema_rag A/B — success rate by category (%)")
ax.set_ylabel("success rate (%)")
ax.set_xlabel("")
ax.set_ylim(0, 105)
ax.legend(loc="lower right")
ax.grid(axis="y", linestyle="--", alpha=0.4)
for label in ax.get_xticklabels():
    label.set_rotation(20)
    label.set_horizontalalignment("right")
plt.tight_layout()
plt.show()

df